# IMLS Downloads Pipeline

The IMLS scraper was not functioning (as of April 2026). IMLS provides a csv download path that allows for all grants to be downloaded. The data can be found at https://www.imls.gov/grants/awarded-grants. While the robots.txt file for imls.gov does not disallows the /grants/* path it is not explicitly allowed. For simplicity we manually download the years of interest to a relevant directory.

This notebook will:

1. Read the designated csv file. This file should be located separate from the final data outputs directory for convenience and clarity.
2. Load the CSV file to a dataframe
3. Process each line into the OIC data model
4. Write out a json-lines format file in the OIC datamodel to the designated data output directory. Output location MUST be a directory to allow for batching of the outputs into multiple files

In [1]:
from pathlib import Path
import csv
from zipfile import ZipFile
import json
from itertools import batched
from attrs import asdict
from csv import DictReader
import polars as pl
from typing import List, Dict, Any
from datetime import datetime, date
from oic_scrape.items import AwardParticipant, AwardItem


In [2]:
DATA_INPUT_DIRECTORY = Path("../data/202604/imls")
FUNDER_ORG_NAME = "Institute of Museum and Library Services"
FUNDER_ROR_ID = "https://ror.org/030prv062"
OUTPUT_LOCATION = Path("../data/202604")
OUTPUT_FORMAT = "jsonl"

In [3]:
awards = []
csv_files = DATA_INPUT_DIRECTORY.glob("*.csv")

for file in csv_files:
    with open(file, "r", encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            award = AwardItem(
                _crawled_at=datetime.now(),
                source='https://www.imls.gov/grants/awarded-grants',
                grant_id=f"imls::{row['Log Number']}",
                funder_org_name=FUNDER_ORG_NAME,
                funder_org_ror_id=FUNDER_ROR_ID,
                recipient_org_name=row['Institution'],
                recipient_org_location=f"{row['City']}, {row['State']}, USA",
                grant_year=int(row['Fiscal Year']),
                award_amount=float(row['Federal Funds'].lstrip('$')),
                award_currency='USD',
                grant_description=row['Description'],
                raw_source_data = json.dumps(','.join(row.values())),
            )
            awards.append(asdict(award))


In [4]:
pl.DataFrame(awards).write_ndjson(OUTPUT_LOCATION / f'imls_grants.jsonl')